<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/Nemo_Moderation_raw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1A - check the GPU

In [3]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

Sat Oct  3 08:53:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1B: Install libraries

In [4]:
!pip install -q -U transformers accelerate pillow huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 58.3 MB/s eta 0:00:00


## Step 1C: Log in to Hugging Face

In [1]:
from huggingface_hub import notebook_login
notebook_login()

## Step 1D: Load the model

In [2]:
import os, torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

MODEL_ID = "nvidia/Nemotron-3.5-Content-Safety"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
).eval()

print("Model loaded on:", model.device)

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/6.60k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/824 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.52k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

Model loaded on: cuda:0


## Step 1E: Add your inference function

In [3]:
PLACEHOLDER = "What is happening in this image?"

def _run_safety_model(image_path=None, prompt=PLACEHOLDER, response=None, categories=True):

    if image_path and not os.path.exists(image_path):
        raise FileNotFoundError(f"{image_path} not found in {os.getcwd()}")

    content = []
    if image_path:
        content.append({"type": "image", "image": image_path})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]

    if response:
        messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": response}]
        })

    kwargs = {"request_categories": "/categories"} if categories else {}

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        **kwargs
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)

    return processor.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

print("Function defined.")

Function defined.


## Step 1G: Add the parser and classify()

In [4]:
import re

def parse_safety_output(raw: str) -> dict:
    """Turn the model's raw text into {'label', 'categories', 'raw'}."""
    label = "review"          # default if we can't parse
    categories = []

    m = re.search(r"User Safety:\s*(safe|unsafe)", raw, re.IGNORECASE)
    if m:
        label = m.group(1).lower()

    c = re.search(r"Safety Categories:\s*(.+)", raw, re.IGNORECASE)
    if c:
        categories = [x.strip() for x in c.group(1).split(",") if x.strip()]

    # If it says unsafe but gave no category, still keep it unsafe.
    return {"label": label, "categories": categories, "raw": raw}


def classify(text=None, image_path=None, response=None):
    """
    Main safety function.
      - text only:         classify(text="...")
      - image only:        classify(image_path="x.jpg")
      - image + caption:   classify(text="caption", image_path="x.jpg")
    """
    if text is None and image_path is None:
        raise ValueError("Provide text, image_path, or both.")

    prompt = text if text else PLACEHOLDER
    raw = _run_safety_model(image_path=image_path, prompt=prompt, response=response)
    return parse_safety_output(raw)

print("classify() ready.")

classify() ready.


## Step 1F: Test with text, then look at the raw output

In [5]:
for t in ["I love hiking in the mountains on weekends.",
          "I am going to find you and hurt you badly."]:
    print(t)
    print(classify(text=t))
    print("-" * 50)

I love hiking in the mountains on weekends.
{'label': 'safe', 'categories': [], 'raw': 'User Safety: safe'}
--------------------------------------------------
I am going to find you and hurt you badly.
{'label': 'unsafe', 'categories': ['Threat'], 'raw': 'User Safety: unsafe\nSafety Categories: Threat'}
--------------------------------------------------


In [7]:
print("Image only:   ", classify(image_path="/content/image.jpg"))
print("Image+caption:", classify(text="Look at this view!", image_path="/content/1_safe.png"))

Image only:    {'label': 'unsafe', 'categories': ['Violence'], 'raw': 'User Safety: unsafe\nSafety Categories: Violence'}
Image+caption: {'label': 'safe', 'categories': [], 'raw': 'User Safety: safe'}


## Step 2A: Add the router

In [8]:
import os, re

EXT_MAP = {
    "image":    {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tiff"},
    "video":    {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"},
    "audio":    {".mp3", ".wav", ".m4a", ".flac", ".ogg", ".aac"},
    "document": {".pdf", ".docx", ".txt", ".csv", ".xlsx", ".md"},
}

URL_RE = re.compile(r"^https?://\S+$", re.IGNORECASE)

PIPELINE = {
    "text":     ["Nemotron-3.5 Content Safety"],
    "image":    ["Nemotron-3.5 Content Safety"],
    "video":    ["Nemotron-3 Nano Omni", "Nemotron-3.5 Content Safety"],
    "audio":    ["Nemotron-3 Nano Omni", "Nemotron-3.5 Content Safety"],
    "document": ["Text extraction", "Nemotron-3.5 Content Safety"],
    "url":      ["Not implemented yet"],
}

def detect_input_type(text=None, file=None, check_exists=True):
    """Return one of: text, image, video, audio, document, url, unknown."""
    if file:
        if check_exists and not os.path.exists(file):
            raise FileNotFoundError(f"{file} not found in {os.getcwd()}")
        ext = os.path.splitext(file)[1].lower()
        for kind, exts in EXT_MAP.items():
            if ext in exts:
                return kind
        return "unknown"

    if text:
        if URL_RE.match(text.strip()):
            return "url"
        return "text"

    raise ValueError("Provide text or file.")

def route(text=None, file=None, check_exists=True):
    kind = detect_input_type(text=text, file=file, check_exists=check_exists)
    return {"input_type": kind, "pipeline": PIPELINE.get(kind, [])}

print("Router ready.")

Router ready.


In [9]:
tests = [
    dict(text="Hello world"),
    dict(text="https://example.com/post/123"),
    dict(file="clip.MP4"),
    dict(file="voice.wav"),
    dict(file="report.pdf"),
    dict(file="data.xlsx"),
    dict(file="archive.zip"),
]

for t in tests:
    print(t, "->", route(check_exists=False, **t))

print("\nReal file:")
print(route(file="/content/1_safe.png"))

print("\nMissing file (should raise an error):")
try:
    route(file="/content/does_not_exist.jpg")
except FileNotFoundError as e:
    print("OK, got expected error:", e)

{'text': 'Hello world'} -> {'input_type': 'text', 'pipeline': ['Nemotron-3.5 Content Safety']}
{'text': 'https://example.com/post/123'} -> {'input_type': 'url', 'pipeline': ['Not implemented yet']}
{'file': 'clip.MP4'} -> {'input_type': 'video', 'pipeline': ['Nemotron-3 Nano Omni', 'Nemotron-3.5 Content Safety']}
{'file': 'voice.wav'} -> {'input_type': 'audio', 'pipeline': ['Nemotron-3 Nano Omni', 'Nemotron-3.5 Content Safety']}
{'file': 'report.pdf'} -> {'input_type': 'document', 'pipeline': ['Text extraction', 'Nemotron-3.5 Content Safety']}
{'file': 'data.xlsx'} -> {'input_type': 'document', 'pipeline': ['Text extraction', 'Nemotron-3.5 Content Safety']}
{'file': 'archive.zip'} -> {'input_type': 'unknown', 'pipeline': []}

Real file:
{'input_type': 'image', 'pipeline': ['Nemotron-3.5 Content Safety']}

Missing file (should raise an error):
OK, got expected error: /content/does_not_exist.jpg not found in /content


**Step 3 : document pipeline**

## Step 3A: Install the document libraries

In [10]:
!pip install -q pypdf python-docx openpyxl pandas fpdf2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 33.7 MB/s eta 0:00:00


## Step 3B: Text extractors

In [11]:
import pandas as pd
from pypdf import PdfReader
from docx import Document
from openpyxl import load_workbook

def extract_document(path):
    """Return a list of (location, text) tuples."""
    ext = os.path.splitext(path)[1].lower()
    segments = []

    if ext in (".txt", ".md", ".csv"):
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            segments.append(("file", f.read()))

    elif ext == ".pdf":
        reader = PdfReader(path)
        for i, page in enumerate(reader.pages, start=1):
            segments.append((f"page {i}", page.extract_text() or ""))

    elif ext == ".docx":
        doc = Document(path)
        parts = [p.text for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                parts.append(" | ".join(cell.text for cell in row.cells))
        segments.append(("document", "\n".join(parts)))

    elif ext == ".xlsx":
        wb = load_workbook(path, read_only=True, data_only=True)
        for ws in wb.worksheets:
            rows = []
            for row in ws.iter_rows(values_only=True):
                cells = [str(c) for c in row if c is not None]
                if cells:
                    rows.append(" | ".join(cells))
            segments.append((f"sheet {ws.title}", "\n".join(rows)))

    else:
        raise ValueError(f"Unsupported document type: {ext}")

    return segments

print("Extractors ready.")

Extractors ready.


## Step 3C: Chunking and document moderation

In [12]:
def chunk_text(text, max_chars=2000):
    """Split text into pieces of at most max_chars, preferring line breaks."""
    chunks, current = [], ""
    for line in text.splitlines():
        while len(line) > max_chars:                 # very long line: hard split
            if current:
                chunks.append(current); current = ""
            chunks.append(line[:max_chars]); line = line[max_chars:]
        if len(current) + len(line) + 1 > max_chars:
            if current:
                chunks.append(current)
            current = line
        else:
            current = current + "\n" + line if current else line
    if current.strip():
        chunks.append(current)
    return [c for c in chunks if c.strip()]


def moderate_document(path, max_chunks=40, max_chars=2000):
    segments = extract_document(path)

    all_chunks = []   # (location, chunk_text)
    for loc, text in segments:
        for c in chunk_text(text, max_chars):
            all_chunks.append((loc, c))

    # Nothing extracted -> probably scanned/image-based (handled in Step 6)
    if not all_chunks:
        return {
            "decision": "REVIEW", "categories": [], "input_type": "document",
            "chunks_checked": 0, "chunks_total": 0, "truncated": False,
            "flagged_chunks": [],
            "note": "No extractable text. Possibly a scanned document (needs OCR, Step 6).",
        }

    total = len(all_chunks)
    to_check = all_chunks[:max_chunks]

    flagged, cats, any_review = [], set(), False
    for i, (loc, chunk) in enumerate(to_check):
        r = classify(text=chunk)
        if r["label"] == "unsafe":
            cats.update(r["categories"])
            flagged.append({"chunk": i, "location": loc,
                            "categories": r["categories"], "preview": chunk[:120]})
        elif r["label"] == "review":
            any_review = True

    truncated = total > max_chunks
    if flagged:
        decision = "UNSAFE"
    elif any_review or truncated:
        decision = "REVIEW"      # not fully checked, so don't claim SAFE
    else:
        decision = "SAFE"

    return {
        "decision": decision, "categories": sorted(cats), "input_type": "document",
        "chunks_checked": len(to_check), "chunks_total": total,
        "truncated": truncated, "flagged_chunks": flagged, "note": "",
    }

print("Document moderation ready.")

Document moderation ready.


## Step 3D: Create test files and run them

In [13]:
from docx import Document as DocxWriter
from fpdf import FPDF

safe_para = "The quarterly meeting covered budgets, hiring plans, and the new office layout. " * 5
bad_line  = "I am going to find you and hurt you badly."

# TXT: safe, and unsafe with the threat in the middle
open("/content/safe.txt", "w").write(safe_para * 20)
open("/content/unsafe.txt", "w").write(safe_para * 10 + "\n" + bad_line + "\n" + safe_para * 10)

# DOCX
d = DocxWriter(); d.add_paragraph(safe_para); d.add_paragraph(bad_line)
d.save("/content/unsafe.docx")

# CSV
pd.DataFrame({"name": ["Ann", "Bob"], "comment": ["Great product", bad_line]}).to_csv("/content/unsafe.csv", index=False)

# XLSX
pd.DataFrame({"name": ["Ann", "Bob"], "comment": ["Nice weather", "See you at lunch"]}).to_excel("/content/safe.xlsx", index=False)

# PDF
pdf = FPDF(); pdf.add_page(); pdf.set_font("Helvetica", size=12)
pdf.multi_cell(0, 8, safe_para + " " + bad_line)
pdf.output("/content/unsafe.pdf")

# Run all
for f in ["safe.txt", "unsafe.txt", "unsafe.docx", "unsafe.csv", "safe.xlsx", "unsafe.pdf"]:
    r = moderate_document("/content/" + f)
    print(f"{f:14} -> {r['decision']:7} cats={r['categories']} "
          f"chunks={r['chunks_checked']}/{r['chunks_total']}")

safe.txt       -> SAFE    cats=[] chunks=4/4
unsafe.txt     -> UNSAFE  cats=['Threat'] chunks=5/5
unsafe.docx    -> UNSAFE  cats=['Threat', 'Violence'] chunks=1/1
unsafe.csv     -> UNSAFE  cats=['Threat', 'Violence'] chunks=1/1
safe.xlsx      -> SAFE    cats=[] chunks=1/1
unsafe.pdf     -> UNSAFE  cats=['Threat', 'Violence'] chunks=1/1


## Step 3E: Testing document

In [14]:
print(moderate_document("/content/unsafe.txt"))

{'decision': 'UNSAFE', 'categories': ['Threat'], 'input_type': 'document', 'chunks_checked': 5, 'chunks_total': 5, 'truncated': False, 'flagged_chunks': [{'chunk': 2, 'location': 'file', 'categories': ['Threat'], 'preview': 'I am going to find you and hurt you badly.'}], 'note': ''}


## Step 4: Test a text call

In [15]:
!pip install -q openai

In [16]:
from google.colab import userdata
from openai import OpenAI

OMNI_MODEL = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning"

omni_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=userdata.get("NVIDIA_API_KEY"),
)

resp = omni_client.chat.completions.create(
    model=OMNI_MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: Omni is working."}],
    max_tokens=512,
    temperature=0,
)

print(resp.choices[0].message.content)

Omni is working.


## Omni image test

In [17]:
import base64, mimetypes

def file_to_data_url(path):
    mime = mimetypes.guess_type(path)[0] or "application/octet-stream"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    return f"data:{mime};base64,{b64}"

resp = omni_client.chat.completions.create(
    model=OMNI_MODEL,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this image in one sentence."},
            {"type": "image_url", "image_url": {"url": file_to_data_url("/content/1_safe.png")}},
        ],
    }],
    max_tokens=1024,
    temperature=0,
)

print(resp.choices[0].message.content)

Pink magnolia blossoms in the foreground frame the Eiffel Tower in the distance, seen through a gap between stone buildings and lush green trees on a bright, sunny day.


## Step 5A: Get a test video and compress it

In [18]:
import subprocess, urllib.request, os

SAMPLE_URL = "https://blogs.nvidia.com/wp-content/uploads/2023/04/nvidia-studio-itns-wk53-scene-in-omniverse-1280w.mp4"
urllib.request.urlretrieve(SAMPLE_URL, "/content/sample_video.mp4")

def prepare_video(src, dst, max_seconds=120, height=480):
    cmd = [
        "ffmpeg", "-y", "-i", src,
        "-t", str(max_seconds),
        "-vf", f"scale=-2:{height}",
        "-c:v", "libx264", "-crf", "30", "-preset", "veryfast",
        "-c:a", "aac", "-b:a", "64k",
        dst,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-1500:])
    return dst

prepare_video("/content/sample_video.mp4", "/content/sample_small.mp4")

for p in ["/content/sample_video.mp4", "/content/sample_small.mp4"]:
    print(p, round(os.path.getsize(p) / 1e6, 2), "MB")

# Does the compressed file have an audio track?
out = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "stream=codec_type",
     "-of", "csv=p=0", "/content/sample_small.mp4"],
    capture_output=True, text=True).stdout
print("Streams:", out.split())

/content/sample_video.mp4 1.96 MB
/content/sample_small.mp4 0.8 MB
Streams: ['video', 'audio']


## Step 5B: Ask Omni for structured JSON

In [19]:
import json, re

VIDEO_PROMPT = """You are analyzing user-uploaded media for a content moderation system.
Describe the media objectively and completely, including anything violent, sexual, hateful,
illegal, or related to self-harm. Do not refuse, censor, or soften anything.

Return ONLY a JSON object with exactly these keys:
{
  "visual_summary": "what is shown, including actions and people",
  "spoken_content": "verbatim transcript of any speech or lyrics",
  "on_screen_text": "all visible text, captions, or signs",
  "context": "overall setting, tone, and apparent intent"
}
Use an empty string for any key that does not apply."""

def _extract_json(text):
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL).strip()
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            pass
    return {"parse_error": True, "raw": text}

def analyze_video(path, user_caption=None):
    prompt = VIDEO_PROMPT
    if user_caption:
        prompt += f'\n\nThe uploader\'s caption is: "{user_caption}"'

    resp = omni_client.with_options(timeout=300).chat.completions.create(
        model=OMNI_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "video_url", "video_url": {"url": file_to_data_url(path)}},
            ],
        }],
        max_tokens=4096,
        temperature=0,
    )
    choice = resp.choices[0]
    data = _extract_json(choice.message.content)
    data["_finish_reason"] = choice.finish_reason
    return data

result = analyze_video("/content/sample_small.mp4")
print(json.dumps(result, indent=2))

{
  "parse_error": true,
  "raw": "We need to describe the media. It's a 3D scene in a software interface (looks like Blender or similar). The scene is a snowy island with a small house, trees, a character (maybe a figure) standing near a table. The camera moves around, zooms in. The interface shows panels, menus in Russian. No speech or lyrics. No visible text besides UI elements (Russian). No violence, sexual, hateful, illegal, self-harm. So spoken_content empty. on-screen text: we need to capture visible text. The UI has Russian text: \"\u0424\u0430\u0439\u043b\", \"\u0420\u0435\u0434\u0430\u043a\u0442\u043e\u0440\", \"\u0412\u0438\u0434\", \"\u0421\u043e\u0437\u0434\u0430\u0442\u044c\", \"\u0421\u043e\u0445\u0440\u0430\u043d\u0438\u0442\u044c\", \"\u042d\u043a\u0441\u043f\u043e\u0440\u0442\", \"\u041d\u0430\u0441\u0442\u0440\u043e\u0439\u043a\u0438\", \"\u041f\u0430\u0440\u0430\u043c\u0435\u0442\u0440\u044b\", \"\u0421\u043b\u043e\u0438\", \"\u041c\u0430\u0442\u0435\u0440\u0438\u04

## Step 5C: Add the video moderation function

In [20]:
import tempfile

def omni_to_text(d, caption=None):
    parts = []
    if caption:
        parts.append(f"Uploader caption: {caption}")
    parts.append(f"Visual summary: {d.get('visual_summary', '')}")
    parts.append(f"Spoken content: {d.get('spoken_content', '')}")
    parts.append(f"On-screen text: {d.get('on_screen_text', '')}")
    parts.append(f"Context: {d.get('context', '')}")
    return "\n".join(parts)[:6000]


def moderate_from_omni(d, caption=None, input_type="video"):
    models = ["Nemotron-3 Nano Omni", "Nemotron-3.5 Content Safety"]

    # Omni failed to give usable JSON, or the reply was cut off
    if d.get("parse_error") or d.get("_finish_reason") == "length":
        return {
            "decision": "REVIEW", "categories": [], "confidence": None,
            "explanation": "Omni output was unparseable or truncated; needs human review.",
            "input_type": input_type, "models_used": models, "omni_output": d,
        }

    checks = [("combined description", omni_to_text(d, caption))]
    for key in ("spoken_content", "on_screen_text"):
        val = (d.get(key) or "").strip()
        if val:
            checks.append((key, val[:3000]))
    if caption:
        checks.append(("caption", caption[:3000]))

    flagged, cats, any_review = [], set(), False
    for name, text in checks:
        r = classify(text=text)
        if r["label"] == "unsafe":
            cats.update(r["categories"])
            flagged.append(f"{name} ({', '.join(r['categories']) or 'no category'})")
        elif r["label"] == "review":
            any_review = True

    if flagged:
        decision = "UNSAFE"
        explanation = "Flagged in: " + "; ".join(flagged)
    elif any_review:
        decision = "REVIEW"
        explanation = "Safety model output could not be parsed for at least one check."
    else:
        decision = "SAFE"
        explanation = "No unsafe content found in the description, speech, on-screen text, or caption."

    return {
        "decision": decision, "categories": sorted(cats), "confidence": None,
        "explanation": explanation, "input_type": input_type,
        "models_used": models, "omni_output": d,
    }


def moderate_video(path, caption=None):
    small = os.path.join(tempfile.gettempdir(), "prepared_" + os.path.basename(path))
    prepare_video(path, small)
    d = analyze_video(small, user_caption=caption)
    return moderate_from_omni(d, caption=caption, input_type="video")

print("Video moderation ready.")

Video moderation ready.


In [21]:
print("TEST 1: real sample video (expect SAFE)")
r1 = moderate_from_omni(result)
print({k: v for k, v in r1.items() if k != "omni_output"})

print("\nTEST 2: fake Omni output with threatening speech (expect UNSAFE)")
fake = {
    "visual_summary": "A man stands in a parking lot at night.",
    "spoken_content": "I am going to find you and hurt you badly.",
    "on_screen_text": "",
    "context": "Tense, aggressive tone.",
}
r2 = moderate_from_omni(fake)
print({k: v for k, v in r2.items() if k != "omni_output"})

print("\nTEST 3: unparseable Omni output (expect REVIEW)")
r3 = moderate_from_omni({"parse_error": True, "raw": "garbage"})
print({k: v for k, v in r3.items() if k != "omni_output"})

TEST 1: real sample video (expect SAFE)
{'decision': 'REVIEW', 'categories': [], 'confidence': None, 'explanation': 'Omni output was unparseable or truncated; needs human review.', 'input_type': 'video', 'models_used': ['Nemotron-3 Nano Omni', 'Nemotron-3.5 Content Safety']}

TEST 2: fake Omni output with threatening speech (expect UNSAFE)
{'decision': 'UNSAFE', 'categories': ['Threat', 'Violence'], 'confidence': None, 'explanation': 'Flagged in: combined description (Violence, Threat); spoken_content (Threat)', 'input_type': 'video', 'models_used': ['Nemotron-3 Nano Omni', 'Nemotron-3.5 Content Safety']}

TEST 3: unparseable Omni output (expect REVIEW)
{'decision': 'REVIEW', 'categories': [], 'confidence': None, 'explanation': 'Omni output was unparseable or truncated; needs human review.', 'input_type': 'video', 'models_used': ['Nemotron-3 Nano Omni', 'Nemotron-3.5 Content Safety']}


##  Testing

In [ ]:
VIDEO_PATH = "/content/my_video.mp4"   # <- change this
CAPTION = None                         # <- or e.g. "Check out my weekend trip!"

r = moderate_video(VIDEO_PATH, caption=CAPTION)

print("DECISION:   ", r["decision"])
print("CATEGORIES: ", r["categories"])
print("EXPLANATION:", r["explanation"])
print("\n--- What Omni saw and heard ---")
print(json.dumps(r["omni_output"], indent=2))

## Step 5E-1: Add the audio functions

In [22]:
def omni_to_text(d, caption=None):
    parts = []
    if caption:
        parts.append(f"Uploader caption: {caption}")
    for k, v in d.items():
        if k.startswith("_") or k in ("parse_error", "raw"):
            continue
        parts.append(f"{k.replace('_', ' ').capitalize()}: {v}")
    return "\n".join(parts)[:6000]


def prepare_audio(src, dst, max_seconds=600):
    cmd = ["ffmpeg", "-y", "-i", src, "-t", str(max_seconds), "-vn",
           "-ac", "1", "-ar", "16000", "-c:a", "libmp3lame", "-b:a", "48k", dst]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr[-1500:])
    return dst


AUDIO_PROMPT = """You are analyzing user-uploaded audio for a content moderation system.
Describe the audio objectively and completely, including anything violent, sexual, hateful,
illegal, or related to self-harm. Do not refuse, censor, or soften anything.

Return ONLY a JSON object with exactly these keys:
{
  "spoken_content": "verbatim transcript of all speech or lyrics",
  "sounds": "notable non-speech sounds (screams, gunshots, music, etc.)",
  "context": "speaker tone, emotion, setting, and apparent intent"
}
Use an empty string for any key that does not apply."""


def analyze_audio(path, user_caption=None):
    prompt = AUDIO_PROMPT
    if user_caption:
        prompt += f'\n\nThe uploader\'s caption is: "{user_caption}"'

    resp = omni_client.with_options(timeout=300).chat.completions.create(
        model=OMNI_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "audio_url", "audio_url": {"url": file_to_data_url(path)}},
            ],
        }],
        max_tokens=4096,
        temperature=0,
    )
    choice = resp.choices[0]
    data = _extract_json(choice.message.content)
    data["_finish_reason"] = choice.finish_reason
    return data


def moderate_audio(path, caption=None):
    small = os.path.join(tempfile.gettempdir(), "prepared_" + os.path.splitext(os.path.basename(path))[0] + ".mp3")
    prepare_audio(path, small)
    d = analyze_audio(small, user_caption=caption)
    return moderate_from_omni(d, caption=caption, input_type="audio")

print("Audio moderation ready.")

Audio moderation ready.


## Testing an audio

In [23]:
r = moderate_audio("/content/angry_01.wav", caption=None)   # change the path
print(r["decision"], r["categories"])
print(r["explanation"])
print(json.dumps(r["omni_output"], indent=2))

InternalServerError: Error code: 503 - {'error': {'message': 'ResourceExhausted: Worker local total request limit reached (16/16)', 'type': 'Service Unavailable', 'code': 503}}

In [24]:
import time
from openai import InternalServerError, RateLimitError, APIConnectionError

def with_retry(fn, *args, tries=6, base_wait=10, **kwargs):
    for attempt in range(1, tries + 1):
        try:
            return fn(*args, **kwargs)
        except (InternalServerError, RateLimitError, APIConnectionError) as e:
            if attempt == tries:
                raise
            wait = base_wait * attempt
            print(f"Server busy (attempt {attempt}/{tries}): {type(e).__name__}. Retrying in {wait}s...")
            time.sleep(wait)

print("Retry helper ready.")

Retry helper ready.


In [25]:
r = with_retry(moderate_audio, "/content/angry_01.wav", caption=None)
print(r["decision"], r["categories"])
print(r["explanation"])
print(json.dumps(r["omni_output"], indent=2))

Server busy (attempt 1/6): InternalServerError. Retrying in 10s...
REVIEW []
Omni output was unparseable or truncated; needs human review.
{
  "parse_error": true,
  "raw": "The user wants a detailed, objective description of the audio content.\nI need to break down the audio into segments based on the distinct scenes or conversations occurring within it.\n\n**Segment 1 (00:00 - 00:15):**\n- **Audio:** A woman is shouting aggressively. \"Excuse me? You fucking call me a prostitute? How fucking dare you speak to me like that? I ain't some fucking fuck!\" She sounds very angry and defensive.\n- **Context:** It sounds like an argument, possibly outdoors or in a large open space given the reverb. The language is explicit and aggressive.\n\n**Segment 2 (00:15 - 00:30):**\n- **Audio:** A different scene. A man says \"What's going on?\" Then another man says \"I'm gonna kick your shit off the trailer, nigga. I ain't worried about you.\" A woman responds \"I know he was playing but he don't ca

## Step 6A: Install the PDF renderer

In [26]:
!pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 79.2 MB/s eta 0:00:00


Step 6B: Add the scanned-document functions

In [28]:
import fitz  # PyMuPDF

PAGE_PROMPT = """You are reading one page of a user-uploaded document for a content moderation system.
Do not refuse, censor, or soften anything, including violent, sexual, hateful, illegal, or self-harm content.

Return ONLY a JSON object with exactly these keys:
{
  "page_text": "verbatim transcription of all text on the page",
  "visual_content": "description of any photos, drawings, charts, or other images on the page",
  "context": "what kind of page this is and its apparent purpose"
}
Use an empty string for any key that does not apply."""


def render_pdf_pages(path, max_pages=10, zoom=1.5):
    outdir = tempfile.mkdtemp(prefix="pages_")
    doc = fitz.open(path)
    total = len(doc)
    paths = []
    for i, page in enumerate(doc):
        if i >= max_pages:
            break
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        p = os.path.join(outdir, f"page_{i+1}.jpg")
        pix.save(p)
        paths.append(p)
    return paths, total


def analyze_page(img_path):
    resp = omni_client.with_options(timeout=300).chat.completions.create(
        model=OMNI_MODEL,
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": PAGE_PROMPT},
                {"type": "image_url", "image_url": {"url": file_to_data_url(img_path)}},
            ],
        }],
        max_tokens=4096,
        temperature=0,
    )
    choice = resp.choices[0]
    data = _extract_json(choice.message.content)
    data["_finish_reason"] = choice.finish_reason
    return data


def moderate_scanned_document(path, max_pages=10):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        pages, total = render_pdf_pages(path, max_pages=max_pages)
    elif ext in EXT_MAP["image"]:
        pages, total = [path], 1
    else:
        raise ValueError(f"Scanned-document mode supports PDF or image files, got {ext}")

    flagged, cats, any_review = [], set(), False
    page_outputs = []

    for i, img in enumerate(pages, start=1):
        # Check 1: the page image itself, straight to Content Safety
        r = classify(image_path=img)
        if r["label"] == "unsafe":
            cats.update(r["categories"])
            flagged.append(f"page {i} image ({', '.join(r['categories']) or 'no category'})")
        elif r["label"] == "review":
            any_review = True

        # Check 2: Omni reads the page, then Content Safety checks the text
        d = with_retry(analyze_page, img)
        page_outputs.append({"page": i, "omni_output": d})

        if d.get("parse_error") or d.get("_finish_reason") == "length":
            any_review = True
            continue

        checks = []
        text = (d.get("page_text") or "").strip()
        vis = (d.get("visual_content") or "").strip()
        if text:
            checks.append(("text", text[:3000]))
        if vis:
            checks.append(("visual content", vis[:3000]))
        for name, t in checks:
            rr = classify(text=t)
            if rr["label"] == "unsafe":
                cats.update(rr["categories"])
                flagged.append(f"page {i} {name} ({', '.join(rr['categories']) or 'no category'})")
            elif rr["label"] == "review":
                any_review = True

    truncated = total > len(pages)
    if flagged:
        decision, explanation = "UNSAFE", "Flagged in: " + "; ".join(flagged)
    elif any_review or truncated:
        decision = "REVIEW"
        explanation = "Not fully verified (parse problem or pages beyond the limit)."
    else:
        decision, explanation = "SAFE", "No unsafe content found in any checked page."

    return {
        "decision": decision, "categories": sorted(cats), "confidence": None,
        "explanation": explanation, "input_type": "document (scanned)",
        "models_used": ["Nemotron-3 Nano Omni", "Nemotron-3.5 Content Safety"],
        "pages_checked": len(pages), "pages_total": total,
        "pages": page_outputs,
    }

print("Scanned-document moderation ready.")

Scanned-document moderation ready.


Step 6C: Test with generated scanned PDFs

In [29]:
from PIL import Image, ImageDraw, ImageFont

def make_scanned_pdf(text, out_path):
    img = Image.new("RGB", (1240, 700), "white")
    d = ImageDraw.Draw(img)
    try:
        font = ImageFont.load_default(size=36)
    except TypeError:
        font = ImageFont.load_default()
    y = 80
    for line in text.split("\n"):
        d.text((60, y), line, fill="black", font=font)
        y += 70
    img.save(out_path, "PDF")

make_scanned_pdf("Meeting notes\nBudget review on Monday.\nBring the quarterly report.", "/content/scan_safe.pdf")
make_scanned_pdf("Meeting notes\nI am going to find you\nand hurt you badly.", "/content/scan_unsafe.pdf")

print("Text extractor on scan_safe.pdf:", extract_document("/content/scan_safe.pdf"))
print()

for f in ["scan_safe.pdf", "scan_unsafe.pdf"]:
    r = moderate_scanned_document("/content/" + f)
    print(f"{f} -> {r['decision']}  cats={r['categories']}  pages={r['pages_checked']}/{r['pages_total']}")
    print("  Explanation:", r["explanation"])
    print("  OCR text   :", r["pages"][0]["omni_output"].get("page_text"))
    print()

Text extractor on scan_safe.pdf: [('page 1', '')]

scan_safe.pdf -> SAFE  cats=[]  pages=1/1
  Explanation: No unsafe content found in any checked page.
  OCR text   : Meeting notes
Budget review on Monday.
Bring the quarterly report.

scan_unsafe.pdf -> UNSAFE  cats=['Threat', 'Violence']  pages=1/1
  Explanation: Flagged in: page 1 text (Violence, Threat)
  OCR text   : Meeting notes
I am going to find you
and hurt you badly.



Testing your own scanned file

In [ ]:
r = moderate_scanned_document("/content/my_scan.pdf")   # change the path
print(r["decision"], r["categories"])
print(r["explanation"])
print(json.dumps(r["pages"], indent=2))

## Cell 9: The final moderate() function

In [ ]:
def moderate(text=None, file=None, caption=None):
    """
    moderate(text="...")
    moderate(file="/content/x.jpg", caption="optional")
    moderate(file="/content/x.mp4")   # video, audio and documents work the same way
    """
    if file and text and caption is None:
        caption = text                      # text next to a file is treated as its caption
    kind = detect_input_type(text=None if file else text, file=file)

    try:
        if kind == "text":
            return moderate_text(text)
        if kind == "url":
            return _result("REVIEW", [], "URL moderation is not implemented yet.", "url", [])
        if kind == "image":
            return moderate_image(file, caption)
        if kind == "video":
            return moderate_video(file, caption)
        if kind == "audio":
            return moderate_audio(file, caption)
        if kind == "document":
            ext = os.path.splitext(file)[1].lower()
            segments = extract_document(file)
            if any(t.strip() for _, t in segments):
                return moderate_document(file, segments)
            if ext == ".pdf":               # no extractable text -> scanned PDF
                return moderate_scanned(file)
            return _result("REVIEW", [], "Document has no extractable text.", "document", [])
        return _result("REVIEW", [], f"Unsupported file type: {os.path.splitext(file)[1]}", "unknown", [])
    except Exception as e:
        # Fail closed: never return SAFE when something broke.
        return _result("REVIEW", [], f"Processing failed: {type(e).__name__}: {e}", kind, [])


def show(r, details=False):
    print("DECISION:   ", r["decision"])
    print("CATEGORIES: ", r["categories"])
    print("INPUT TYPE: ", r["input_type"])
    print("MODELS:     ", ", ".join(r["models_used"]))
    print("EXPLANATION:", r["explanation"])
    if details:
        print(json.dumps(r["details"], indent=2, default=str))

print("moderate() ready.")

## Cell 10: Your own test cell

In [ ]:
# Text
show(moderate(text="Your text here"))

# Any file: image, video, audio, pdf, docx, txt, csv, xlsx, scanned pdf
# show(moderate(file="/content/your_file.mp4"), details=True)

# File with a caption
# show(moderate(file="/content/photo.jpg", caption="My caption"))